# Parte 2 - Del LLM al clasificador

## 2.1. Línea base Zero-Shot (sin fine-tuning)

**Enunciado**: 
- Evaluar 40 titulares usando pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli") o Recognai/bert-base-spanish-wwm-cased-xnli.
- Reportar accuracy y F1 macro. Este puntaje representa el piso mínimo a superar.

Antes de iniciar, se instaló transformers + torch + accelerate, datasets, scikit-learn, seaborn, google-genai, python-dotenv, y se registró la clave de Gemini, la IA que se usará.

In [1]:
import os
from dotenv import load_dotenv
import transformers, torch

load_dotenv()  # lee el archivo .env
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("¿Clave de Gemini cargada?", os.getenv("GEMINI_API_KEY") is not None)

c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.20.1 | torch 2.14.1+cpu
¿Clave de Gemini cargada? True


Se verifica que se cargó correctamente la clave de Gemini.

Ahora, el zero-shot es un modelo ya entrenado para decidir si una frase encaja o no con una etiqueta escrita en palabras (explicado por Claude). 

Para la prueba, como indica la tarea, se tomarán 40 titulares, 10 por cada categoría. 

In [ ]:
import pandas as pd

noticias = pd.read_csv("datos/noticias.csv")

# 10 titulares al azar de cada categoría (semilla fija para que siempre salgan los mismos)
muestra_zs = noticias.groupby("etiqueta").sample(10, random_state=42)
print(muestra_zs.shape)
muestra_zs["etiqueta"].value_counts()

(40, 7)


etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64

Una vez seleccionados los 40 titulares, se procede a aplicar el modelo zero-shot. Para ello, se determinó que se pruebe con "Esta noticia trata sobre..." política/economía/deportes/ciencia y tecnología.

In [3]:
from transformers import pipeline

clasificador_zs = pipeline("zero-shot-classification",
                           model="Recognai/bert-base-spanish-wwm-cased-xnli")

# El modelo necesita las categorías escritas en palabras; luego se traducen a las etiquetas oficiales
CATEGORIAS = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}

resultados = clasificador_zs(list(muestra_zs["titular"]),
                             candidate_labels=list(CATEGORIAS),
                             hypothesis_template="Esta noticia trata sobre {}.")

# Para cada titular, la categoría con mayor puntaje
muestra_zs["prediccion_zs"] = [CATEGORIAS[r["labels"][0]] for r in resultados]


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 160.87it/s]


Ahora, se calculan las métricas: accuracy (porcentaje de aciertos) y F1 macro (promedio del desempeño en cada categoría).

In [4]:
from sklearn.metrics import accuracy_score, f1_score

acc_zs = accuracy_score(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"])
f1_zs = f1_score(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"], average="macro")
print(f"Zero-shot -> Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f}")

pd.crosstab(muestra_zs["etiqueta"], muestra_zs["prediccion_zs"])

Zero-shot -> Accuracy: 0.575 | F1 macro: 0.560


prediccion_zs,ciencia_tecnologia,deportes,economia,politica
etiqueta,,,,
ciencia_tecnologia,9,1,0,0
deportes,0,5,0,5
economia,1,1,2,6
politica,0,1,2,7


Los valores de accuracy son de 0,575, y de F1 macro es 0,560, los cuales son bajos.

Se identifica que en ciencia_tecnologia acertó en casi todos (9 de 10), pero en deportes, economía y política, los aciertos bajan (5, 2 y 7 de 10, respectivamente). 

Además, se identifica que el modelo tiende a clasificar las noticias de deportes como política (5 de 10), así como de de economía como política (6 de 10).

Frente a ello, se espera que este método base debe ser superado por los otros métodos.

## 2.2 Supervisión débil con LLM

**Enunciado**: 
- Usar un LLM (Ollama, Qwen local, Gemini API o similar) con un prompt estructurado para etiquetar el dataset completo en las 4 categorías.
- Guardar datos/etiquetas_llm.csv con: titular, etiqueta_llm, confianza, modelo_llm, fecha.
- Auditoría humana: Revisar manualmente 20 predicciones al azar y reportar el porcentaje de acuerdo (ej. 18/20 = 90%).

Para la supervisión, se urará Gemini API. Para ello, se va a configurar la búsqueda, limitando exclusivamente de las 4 categorías definidas, así como determinar qué información extraer. Además, debido a que gemini-2.5-flash ya no está disponible para cuentas nuevas, se va usar gemini-3.8-flash.

In [ ]:
import time
from datetime import date
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.genai import types

MODELO_LLM = "gemini-3.8-flash"
cliente = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Formato de respuesta obligatorio: Gemini debe devolver exactamente esto por cada titular
class Clasificacion(BaseModel):
    id: int
    etiqueta: Literal["politica", "economia", "deportes", "ciencia_tecnologia"]
    confianza: float

INSTRUCCIONES = """Eres editor de un diario peruano. Clasifica cada titular en UNA de estas 4 categorías:
- politica: gobierno, Congreso, elecciones, partidos, justicia y casos policiales.
- economia: empresas, precios, empleo, impuestos, finanzas y comercio.
- deportes: fútbol y otras disciplinas deportivas.
- ciencia_tecnologia: ciencia, salud, medio ambiente, espacio y tecnología.
Para cada titular devuelve su número (id), la categoría y tu confianza entre 0 y 1."""

def clasificar_lote(lote):
    """Envía un grupo de titulares a Gemini y devuelve su clasificación."""
    lineas = "\n".join(f"{i}. {t}" for i, t in zip(lote.index, lote["titular"]))
    respuesta = cliente.models.generate_content(
        model=MODELO_LLM,
        contents=INSTRUCCIONES + "\n\nTitulares:\n" + lineas,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=list[Clasificacion],
        ),
    )
    return respuesta.parsed

Ahora, se van a implementar la clasificación de las 399 noticias en lotes de 20.

In [6]:
TAMANO_LOTE = 20
filas = []

for inicio in range(0, len(noticias), TAMANO_LOTE):
    lote = noticias.iloc[inicio:inicio + TAMANO_LOTE]
    resultado = []
    for intento in range(2):  # máximo 1 reintento
        try:
            resultado = clasificar_lote(lote)
            break
        except Exception as error:
            print("Error, se reintenta en 30 s:", error)
            time.sleep(30)
    for c in resultado:
        filas.append({"id": c.id, "etiqueta_llm": c.etiqueta, "confianza": c.confianza})
    print(f"Lote {inicio // TAMANO_LOTE + 1}: {len(resultado)} titulares")
    time.sleep(5)  # pausa para respetar el límite gratuito de pedidos por minuto

Lote 1: 20 titulares
Lote 2: 20 titulares
Lote 3: 20 titulares
Lote 4: 20 titulares
Lote 5: 20 titulares
Lote 6: 20 titulares
Lote 7: 20 titulares
Lote 8: 20 titulares
Lote 9: 20 titulares
Lote 10: 20 titulares
Lote 11: 20 titulares
Lote 12: 20 titulares
Lote 13: 20 titulares
Lote 14: 20 titulares
Lote 15: 20 titulares
Lote 16: 20 titulares
Lote 17: 20 titulares
Lote 18: 20 titulares
Lote 19: 20 titulares
Lote 20: 19 titulares


Se logró clasificar (en 20 lotes de 20) las 399 noticias. Ahora se procede a armar los datos y guardarlos en un archivo csv.

In [7]:
llm = pd.DataFrame(filas).set_index("id")
llm = llm[~llm.index.duplicated()]  # por si Gemini repitiera algún id

etiquetas_llm = noticias[["titular"]].join(llm)  # une por el número de cada noticia
etiquetas_llm["modelo_llm"] = MODELO_LLM
etiquetas_llm["fecha"] = date.today().isoformat()  # fecha del etiquetado

print("Titulares sin etiqueta del LLM:", etiquetas_llm["etiqueta_llm"].isna().sum())
etiquetas_llm.to_csv("datos/etiquetas_llm.csv", index=False, encoding="utf-8-sig")
etiquetas_llm.head()


Titulares sin etiqueta del LLM: 0


,titular,etiqueta_llm,confianza,modelo_llm,fecha
0,Alejandro Aguinaga viajó a Argentina el martes...,politica,0.98,gemini-3.8-flash,2026-10-18
1,Álvaro Vargas Llosa niega intromisión política...,politica,0.96,gemini-3.8-flash,2026-10-18
2,Solicitan la destitución de Roberto Burneo por...,politica,0.99,gemini-3.8-flash,2026-10-18
3,Puno: alcalde de Quiaca muere tras caer a un b...,politica,0.91,gemini-3.8-flash,2026-10-18
4,Ministro Rafael Belaúnde propone que las FF.AA...,politica,0.98,gemini-3.8-flash,2026-10-18


In [8]:
acc_llm = accuracy_score(noticias["etiqueta"], etiquetas_llm["etiqueta_llm"])
f1_llm = f1_score(noticias["etiqueta"], etiquetas_llm["etiqueta_llm"], average="macro")
print(f"Gemini vs. sección -> Accuracy: {acc_llm:.3f} | F1 macro: {f1_llm:.3f}")
pd.crosstab(noticias["etiqueta"], etiquetas_llm["etiqueta_llm"])


Gemini vs. sección -> Accuracy: 0.955 | F1 macro: 0.955


etiqueta_llm,ciencia_tecnologia,deportes,economia,politica
etiqueta,,,,
ciencia_tecnologia,100,0,0,0
deportes,0,99,0,0
economia,4,0,88,8
politica,3,0,3,94


Se verifica que todos los titulares tienen el puntaje de accuracy y  el F1 macro de 0,955 cada uno. Eso se evidencia en la tabla, donde en ciencia_tecnologia y deportes tienen una coincidencia del total de noticias; sin embargo, en economía y política sí hubo errores. En el primero, 4 noticias se etiquetaron como iencia_tecnologia, y 8 como política. En el segundo, 3 se etiquetaron como ciencia_tecnologia y 3 como economía. 

In [9]:
pd.set_option("display.max_colwidth", None)  # para ver los titulares completos
auditoria = etiquetas_llm.sample(20, random_state=42)[["titular", "etiqueta_llm", "confianza"]]
auditoria


,titular,etiqueta_llm,confianza
198,Corpac: Nueva torre de control del aeropuerto Jorge Chávez estará lista en enero,economia,0.85
349,"Franco Velazco dice estar ""curtido"" ante carta notarial de Alianza Lima: ""Tengo cuero de chancho""",deportes,0.97
33,Halloween y Día de la Canción Criolla: pymes esperan elevar sus ventas hasta 12% en octubre,economia,0.98
208,"Mano Menezes descartó que desconvocaran a Oliver Sonne por lesión: ""Anticipamos el regreso con su club""",deportes,1.00
93,APP: aportantes del partido de Acuña obtuvieron contratos en el Ministerio de Trabajo,politica,0.95
84,Fiscalía tiene en sus manos caso de contratos de López Aliaga con abogado Martín De Luca,politica,0.98
374,Renzo Manyari será nuestro representante ante la Asociación de Comités Olímpicos Nacionales,deportes,0.99
94,GORE de La Libertad pagó más de S/11 mil a César Acuña por vacaciones truncas tras su fallida candidatura presidencial,politica,0.92
222,"Programación de la fecha 1 de la Liga Peruana de Vóley 2026-27: partidos, horarios y canal confirmado para ver el torneo",deportes,1.00
126,Día del Shopping movería S/320 millones: ¿qué productos concentrarán las compras este fin de semana?,economia,0.98


Solo leyendo los titulres, se identifica que casi todos los casos acertó en la etiquea. El único caso que tenía duda era la noticia 198, ya que no estaba segura del todo si corresponde a una nota de economía. Sin embargo, al revisarlo en la página de La República, sí corresponde a una noticia de economía. 

Por lo tanto, coincido con el puntaje atribuido a casa titular.

In [11]:
# Revisión manual: 1 = de acuerdo con Gemini, 0 = en desacuerdo (en el orden de la tabla)
revision_humana = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
auditoria["acuerdo_humano"] = revision_humana
print(f"Acuerdo LLM-humano: {sum(revision_humana)}/20 = {sum(revision_humana) / 20:.0%}")

Acuerdo LLM-humano: 20/20 = 100%


## 2.3. Partición de datos (Split 70 / 15 / 15)

**Enunciado**:

- Realizar split estratificado con random_state=42 (Train ~70%, Val ~15%, Test ~15%). 
- El conjunto de Test debe permanecer estrictamente aislado hasta la evaluación final.

Se va a dividir en 3 grupos: el entrenamiento (70%), el modelo aprende a clasificar los titulares; la validación (15%), se va a revisar cómo va el aprendizaje, para elegir la mejor versión del modelo; por último, el test (15%) se usa una sola vez, al final, para evaluar la versión final.

In [12]:
from sklearn.model_selection import train_test_split

# 1) 70% train y 30% "resto"
train, resto = train_test_split(noticias, test_size=0.30,
                                stratify=noticias["etiqueta"], random_state=42)

# 2) El 30% restante se parte en dos mitades: 15% validación y 15% test
val, test = train_test_split(resto, test_size=0.50,
                             stratify=resto["etiqueta"], random_state=42)

print("Train:", len(train), "| Validación:", len(val), "| Test:", len(test))

# Se comprueba que las 4 categorías quedaron en la misma proporción en cada grupo
pd.DataFrame({
    "train": train["etiqueta"].value_counts(),
    "validacion": val["etiqueta"].value_counts(),
    "test": test["etiqueta"].value_counts(),
})

Train: 279 | Validación: 60 | Test: 60


,train,validacion,test
etiqueta,,,
ciencia_tecnologia,70,15,15
deportes,69,15,15
economia,70,15,15
politica,70,15,15


Se va a realizar el entrenamiento con 279 noticias, y se entrenará a partir de la columna etiqueta, referente a la sección del diario La República. Ya se determinó que las etiquetas de Gemini coinciden en un 95.5%, así que podrían servir para entrenar si no se tuviera la sección.

## 2.4. Fine-Tuning de Transformer

**Enunciado**:

- Modelo recomendado: distilbert-base-multilingual-cased o dccuchile/bert-base-spanish-wwm-cased (BETO).
- Configuración mínima con HuggingFace Trainer: 3 épocas, batch size de 8 a 16, evaluación porépoca y métrica de optimización
F1 macro (sklearn.metrics.f1_score(average="macro")).
- Guardar el modelo en modelo/.

De acuerdo con Claude, el fine-tuning de DistilBERT ya "sabe" leer en muchos idiomas, pero no sabe sobre las etiquetas de esta tarea, por lo cual, se le va agregar una "capa de salida" nueva, para entrenar las 279 noticias y las clasifique en las 4 catogorías definidas. De acuerdo con lo indicado en la tarea, de se debe realizar 3 veces.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODELO_BASE = "distilbert-base-multilingual-cased"
ETIQUETAS = ["ciencia_tecnologia", "deportes", "economia", "politica"]
id2label = dict(enumerate(ETIQUETAS))               # 0 -> "ciencia_tecnologia", 1 -> "deportes", ...
label2id = {e: i for i, e in enumerate(ETIQUETAS)}  # "ciencia_tecnologia" -> 0, ...

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(ETIQUETAS), id2label=id2label, label2id=label2id)

def a_dataset(tabla):
    """Convierte una tabla de noticias al formato del Trainer, con el texto ya tokenizado."""
    datos = Dataset.from_dict({
        "text": tabla["titular"].tolist(),
        "label": tabla["etiqueta"].map(label2id).tolist(),
    })
    return datos.map(lambda x: tokenizador(x["text"], truncation=True, max_length=64), batched=True)

ds_train = a_dataset(train)
ds_val = a_dataset(val)
ds_test = a_dataset(test)  # se prepara ahora, pero NO se usa hasta el paso 5

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 300.65it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 60/60 [00:00<00:00, 2910.29 examples/s]


Ahora, se va configurar y entrenar el modelo.

In [14]:
def calcular_metricas(prediccion):
    """Accuracy y F1 macro, para evaluar cada época en validación."""
    reales = prediccion.label_ids
    predichas = np.argmax(prediccion.predictions, axis=1)
    return {"accuracy": accuracy_score(reales, predichas),
            "f1_macro": f1_score(reales, predichas, average="macro")}

argumentos = TrainingArguments(
    output_dir="modelo",               # aquí se guardan las copias intermedias (checkpoints)
    num_train_epochs=3,                # 3 épocas, como pide el enunciado
    per_device_train_batch_size=16,    # 16 titulares por paso (el enunciado pide entre 8 y 16)
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    eval_strategy="epoch",             # evalúa en validación al terminar cada época
    save_strategy="epoch",
    load_best_model_at_end=True,       # al final se queda con la mejor época…
    metric_for_best_model="f1_macro",  # …según el F1 macro
    save_total_limit=1,                # guarda solo 1 copia intermedia (cada una pesa ~1,6 GB)
    seed=42,
    report_to="none",
)

entrenador = Trainer(model=modelo, args=argumentos,
                     train_dataset=ds_train, eval_dataset=ds_val,
                     processing_class=tokenizador, compute_metrics=calcular_metricas)
entrenador.train()


c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_1\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.803964,0.866667,0.866774
2,No log,0.405220,0.883333,0.882703
3,No log,0.325746,0.883333,0.882076


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.03s/it]
c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_1\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]
c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_1\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]


TrainOutput(global_step=54, training_loss=0.6596346961127387, metrics={'train_runtime': 261.8071, 'train_samples_per_second': 3.197, 'train_steps_per_second': 0.206, 'total_flos': 7872776895648.0, 'train_loss': 0.6596346961127387, 'epoch': 3.0})

El accuracy y F1 macro tiene un puntaje sube de 0,867 (época 1) a 0,883 (épocas 2 y 3); son "épocas", no "intentos", no hay mucha diferencia entre las 3 épocas. Se va a guardar el mejor resultado, el cual corresponde a la época 2 (F1 0,8827), apenas por encima de la 3 (0,8821).

In [15]:
entrenador.save_model("modelo")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.41s/it]
